## Write Urself Challenge
this is a section where I have to rewrite everything from memory and figure it out myself

In [61]:
import math
import random

In [62]:
class Value:
    def __init__(self, num, _child=(), label=''):
        self.data   = num
        self.label  = label

        self._prev  = set(_child)
        self._backward = lambda: None
        self.grad   = 0.0

    # print
    def __repr__(self):
        return f"Value({self.label}:{self.data})"


    # basic operations
    def __add__ (self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad 
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad  += other.data * out.grad
            other.grad +=  self.data * out.grad 
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += other * (self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1 

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def exp(self):
        out = Value(math.exp(self.data), (self,))

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    # activation func
    def sigmoid(self):
        out = Value(1 / (1 + (-self).exp().data), (self,))

        def _backward():
            self.grad += out.data * (1 - out.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self,))

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        t = max(0, self.data)
        out = Value(t, (self,))
        
        def _backward():
            self.grad += (1 if t > 0 else 0) * out.grad
        out._backward = _backward

        return out

    # backprop
    def backward(self):
        topo=[]
        vis =set()

        def build_topo(val):
            if val not in vis:
                vis.add(val)
                for c in val._prev:     
                    build_topo(c)
                topo.append(val)

        build_topo(self)
        self.grad = 1.0
        for val in reversed(topo):
            val._backward()

In [59]:
a = Value(4.0, label='a');
b = Value(3.0, label='b')
c = Value(1.0, label='c')

d = a * b; d.label='d'
e = d + c; e.label='e'
s = e.relu()

s.backward()
s


Value(:13.0)

In [60]:
print(s.grad)
print(e.grad)
print(d.grad)
print(c.grad)
print(b.grad)
print(a.grad)

1.0
1.0
1.0
1.0
4.0
3.0


### NEURAL NETWORK I AM COMING BABYYYY

In [ ]:
class Neuron:
    def __init__(self, nin, activation=None):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b =  Value(random.uniform(-1,1))

        acts = {
            'relu': lambda x: x.relu(),
            'sigmoid': lambda x: x.sigmoid(),
            'tanh': lambda x: x.tanh(),
            'identity': lambda x: x,
        }
        self.activation = acts[activation]

    def __call__(self, x):
        out = sum((xi*wi for xi, wi in zip(x, self.w)), self.b)
        return self.activation(out)

    def parameters(self):
        return [self.b] + self.w

In [106]:
class Layer:
    def __init__(self, nin, nout, activation=None):
        self.neurons = [Neuron(nin, activation=activation) for _ in range(nout)] 

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [ ]:
class MLP:
    def __init__(self, nin, nouts):
        self.layers = []
        prev = nin

        for nout, act in nouts:
            self.layers.append(Layer(nin, nout, activation=act))
            prev = nout

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def train(self, X, y, lr, epochs, batch_size=None, mode='sgd'):
        for _ in range(epochs):
            ypred = [self(x) for x in X]
            loss = sum((yout - ygt)**2 for yout, ygt in zip(ypred, y))

            # backward
            for p in n.parameters():
                p.grad = 0.0
            loss.backward()

            if _ % 50 == 0:
                print(f"{_}: {loss}")

            # forward
            for p in n.parameters():
                p.data += -lr * p.grad

    def predict(self, X):
        return [self(x) for x in X]

In [446]:
# 8 samples with 4 inputs each
xs = [
    [ 1.5,  2.0,  1.0,  0.5], # (1.5*2.0) - (1.0^2) + 0.5 = 3.0 - 1.0 + 0.5 = +2.5 -> 1.0
    [ 2.0, -1.0,  1.5, -0.5], # (2.0*-1.0) - (1.5^2) - 0.5 = -2.0 - 2.25 - 0.5 = -4.75 -> 0.0
    [-1.0, -2.0,  1.0, -0.5], # (-1.0*-2.0) - (1.0^2) - 0.5 = 2.0 - 1.0 - 0.5 = +0.5 -> 1.0
    [ 0.5,  1.0,  2.0,  1.0], # (0.5*1.0) - (2.0^2) + 1.0 = 0.5 - 4.0 + 1.0 = -2.5 -> 0.0
    [ 2.0,  2.0,  0.5, -1.0], # (2.0*2.0) - (0.5^2) - 1.0 = 4.0 - 0.25 - 1.0 = +2.75 -> 1.0
    [-1.5,  1.0,  1.0,  0.0], # (-1.5*1.0) - (1.0^2) + 0.0 = -1.5 - 1.0 + 0.0 = -2.5 -> 0.0
    [ 1.0,  3.0,  1.2, -0.2], # (1.0*3.0) - (1.2^2) - 0.2 = 3.0 - 1.44 - 0.2 = +1.36 -> 1.0
    [-2.0,  0.5,  1.5, -1.0], # (-2.0*0.5) - (1.5^2) - 1.0 = -1.0 - 2.25 - 1.0 = -4.25 -> 0.0
]

ys = [1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0]

In [447]:
# 4 inputs, 2 hidden layers with ReLU, 1 output neuron with Sigmoid
n = MLP(4, [
    (6, 'relu'),
    (6, 'relu'),
    (1, 'sigmoid')
])

n.train(xs, ys, 0.01, 1000)

0: Value(:2.61468284435059)
50: Value(:1.2296470663087553)
100: Value(:0.9957654986661127)
150: Value(:0.8959010677731806)
200: Value(:0.8586469056246009)
250: Value(:0.8382950793551927)
300: Value(:0.8265563888127783)
350: Value(:0.8196025857073628)
400: Value(:0.8139450272791925)
450: Value(:0.8106454217148767)
500: Value(:0.8084527382857122)
550: Value(:0.8067460237812758)
600: Value(:0.8055944507270912)
650: Value(:0.8046649672946294)
700: Value(:0.8040182122648515)
750: Value(:0.8034783956562245)
800: Value(:0.8030208521046764)
850: Value(:0.8027163072114057)
900: Value(:0.8024046720474372)
950: Value(:0.8021763227500719)


In [453]:
x_test = [[1.8, 1.5, 1.1, -0.5]]

n.predict(x_test)

[Value(:0.7617221369528865)]

In [450]:
ypred = [n(x) for x in xs]

for ygt, yout in zip(ys, ypred):
    prob = yout.data
    pred = 1.0 if prob >= 0.5 else 0
    print(f"Target: {ygt} | Prob: {prob:.4f} | Predicted: {pred}")

Target: 1.0 | Prob: 0.7950 | Predicted: 1.0
Target: 0.0 | Prob: 0.0287 | Predicted: 0
Target: 1.0 | Prob: 0.7950 | Predicted: 1.0
Target: 0.0 | Prob: 0.7950 | Predicted: 1.0
Target: 1.0 | Prob: 0.7950 | Predicted: 1.0
Target: 0.0 | Prob: 0.0321 | Predicted: 0
Target: 1.0 | Prob: 0.7950 | Predicted: 1.0
Target: 0.0 | Prob: 0.0035 | Predicted: 0


In [452]:
x_test = [1.8, 1.5, 1.1, -0.5]

pred = n(x_test)
prob = pred.data
print(f"Prob: {prob:.4f} | Predicted: {1.0 if prob >= 0.5 else 0.0}")

Prob: 0.7617 | Predicted: 1.0
